# Feature Engineering
Prepare enriched ads copy csv to be used for ML training. Main steps listed below:
1. Clean empty columns (notably 'placement' was empty from EDA 2)
2. Engineer text features (text lenghts, capitalization, basic urgency and sentiment) NOTE: sentiment and urgency can be enhanced via AI later, currently using simplest search
3. Engineer categorical features (encode categories via LabelEncoder)
4. Engineer performance features (CTR, CVR, high performer, etc. based on clicks, impressions, conversions, and spend)
5. Create interaction features between existing features (i.e. headline length x CTR, platform + ad format, etc.)

*Update: compute_metrics.py is no longer necessary as feature engineering has been compiled together into features.py*

In [1]:
# Run feature engineering script
# Input: initial csv path, Output: path to engineered csv
# Note: Script can use enriched csv *without* metrics, as features.py handles performance metrics as well
import sys
import os
module_path = os.path.join(os.getcwd(), '..', 'source_code', 'app')
if module_path not in sys.path:
    sys.path.append(module_path)

from features import engineer_features

df, feature_list = engineer_features(
    input_csv_path='../data/enriched_ads_with_metrics.csv',
    output_csv_path='../data/ml_ready_ad_features.csv'
)

Loading data from: ../data/enriched_ads_with_metrics.csv
Original shape: (7000, 18)
Dropping empty columns: ['placement']
After dropping empty columns: (7000, 17)
Engineering text features...


c:\Users\kevin.jin\MyGithubProjects\llm_ad_copy_optimizer\notebooks\..\source_code\app\features.py:89: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['headline_urgency'] = df['headline_text'].str.contains(urgency_words, case=False).fillna(False).astype(int)
c:\Users\kevin.jin\MyGithubProjects\llm_ad_copy_optimizer\notebooks\..\source_code\app\features.py:90: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['body_urgency'] = df['body_text'].str.contains(urgency_words, case=False).fillna(False).astype(int)
c:\Users\kevin.jin\MyGithubProjects\llm_ad_copy_optimizer\notebooks\..\source_code\app\features.py:94: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['headline_positive'] = df['headline_text'].str.contains(positive_word

Engineering categorical features...
Engineering performance features...
Creating interaction features...
Selecting final features...
⚠️  Removing non-numeric columns: ['ad_id']
Saving engineered features to: ../data/ml_ready_ad_features.csv
Final shape: (7000, 220)
Final shape: (7000, 220)


In [2]:
# Display results summary
print(f"✅ Feature engineering completed successfully!")
print(f"📊 Final dataset shape: {df.shape}")
print(f"🔧 Total features: {len(feature_list)}")
print(f"\n📁 Output saved to: ../data/ml_ready_ad_features.csv")

# Properly categorize features without overlaps
all_features = set(feature_list)

# Define mutually exclusive categories
pure_text_features = set([f for f in feature_list if any(keyword in f for keyword in 
                ['len', 'word_count', 'caps', 'numbers', 'percent', 'urgency', 'positive', 'exclamation', 'question'])
                and not f.endswith('_encoded') and '_x_' not in f])

pure_categorical_features = set([f for f in feature_list if ('_encoded' in f or 
                any(f.startswith(prefix) for prefix in ['platform_', 'ad_format_', 'category_']))
                and not any(keyword in f for keyword in ['len', 'word_count', 'caps', 'numbers', 'percent', 'urgency', 'positive', 'exclamation', 'question'])])

pure_performance_features = set([f for f in feature_list if f in ['impressions', 'clicks', 'conversions', 'spend', 'ctr', 'cvr', 'cpc', 'cpa']])

pure_engineered_features = set([f for f in feature_list if any(keyword in f for keyword in 
                      ['revenue_efficiency', 'cost_per_thousand', 'engagement_score', 'quartile'])
                      or ('_x_' in f and any(kw in f for kw in ['len', 'cvr', 'ctr']))])

# ID and target features
id_target_features = set(['ad_id', 'high_performer'])

# Any remaining features
other_features = all_features - pure_text_features - pure_categorical_features - pure_performance_features - pure_engineered_features - id_target_features

print(f"\n📋 Feature Breakdown (mutually exclusive):")
print(f"   • Text features: {len(pure_text_features)}")
print(f"   • Categorical features: {len(pure_categorical_features)}")
print(f"   • Performance metrics: {len(pure_performance_features)}")
print(f"   • Engineered features: {len(pure_engineered_features)}")
print(f"   • ID & Target: {len(id_target_features)}")
print(f"   • Other: {len(other_features)}")
print(f"   • Total: {len(pure_text_features) + len(pure_categorical_features) + len(pure_performance_features) + len(pure_engineered_features) + len(id_target_features) + len(other_features)}")

print(f"\n🎯 Ready for ML training!")

✅ Feature engineering completed successfully!
📊 Final dataset shape: (7000, 220)
🔧 Total features: 220

📁 Output saved to: ../data/ml_ready_ad_features.csv

📋 Feature Breakdown (mutually exclusive):
   • Text features: 24
   • Categorical features: 181
   • Performance metrics: 8
   • Engineered features: 5
   • ID & Target: 2
   • Other: 0
   • Total: 220

🎯 Ready for ML training!


## Quick EDA of Engineered Features
Before moving to ML training, let's explore some key patterns in our engineered dataset.

In [4]:
# Basic dataset info
print("📊 Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"High performers: {df['high_performer'].sum()} ({df['high_performer'].mean():.1%})")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print(f"\n🎯 Target Distribution:")
print(df['high_performer'].value_counts().sort_index())

📊 Dataset Overview:
Shape: (7000, 220)
High performers: 1750 (25.0%)
Memory usage: 3.0 MB

🎯 Target Distribution:
high_performer
0    5250
1    1750
Name: count, dtype: int64


In [5]:
# Text features analysis
text_cols = ['headline_len', 'body_len', 'cta_len', 'headline_word_count', 'body_word_count']
print("📝 Text Features Summary:")
print(df[text_cols].describe().round(1))

# Compare high vs low performers
print(f"\n🏆 Text Features: High Performers vs Others:")
for col in text_cols:
    high_perf = df[df['high_performer']==1][col].mean()
    low_perf = df[df['high_performer']==0][col].mean()
    print(f"{col:20}: High={high_perf:.1f}, Low={low_perf:.1f}, Diff={high_perf-low_perf:+.1f}")

📝 Text Features Summary:
       headline_len  body_len  cta_len  headline_word_count  body_word_count
count        7000.0    7000.0   7000.0               7000.0           7000.0
mean           30.6      74.7      6.8                  5.1             11.8
std            16.9      61.7      9.6                  2.9              9.6
min             2.0       0.0      0.0                  1.0              0.0
25%            18.0      16.0      0.0                  3.0              3.0
50%            28.0      70.0      4.5                  5.0             11.0
75%            39.0     116.0     10.0                  6.0             18.0
max           100.0     300.0    118.0                 25.0             61.0

🏆 Text Features: High Performers vs Others:
headline_len        : High=30.9, Low=30.5, Diff=+0.4
body_len            : High=74.0, Low=75.0, Diff=-1.0
cta_len             : High=6.9, Low=6.7, Diff=+0.2
headline_word_count : High=5.2, Low=5.0, Diff=+0.1
body_word_count     : High=11

In [6]:
# Performance metrics patterns
perf_cols = ['ctr', 'cvr', 'cpc', 'cpa', 'engagement_score']
print("⚡ Performance Metrics Summary:")
print(df[perf_cols].describe().round(4))

# Check for any extreme outliers
print(f"\n🚨 Potential Outliers (>99th percentile):")
for col in perf_cols:
    q99 = df[col].quantile(0.99)
    outliers = (df[col] > q99).sum()
    print(f"{col:20}: {outliers} values > {q99:.4f}")

⚡ Performance Metrics Summary:
             ctr        cvr        cpc        cpa  engagement_score
count  7000.0000  7000.0000  7000.0000  7000.0000         7000.0000
mean      0.0173     0.0884     0.8396    14.4025            0.0457
std       0.0047     0.0374     0.4946    20.4073            0.0166
min       0.0070     0.0111     0.1436     0.7744            0.0097
25%       0.0139     0.0620     0.5049     4.8275            0.0344
50%       0.0169     0.0821     0.7751    10.1653            0.0426
75%       0.0200     0.1018     1.0430    15.1981            0.0525
max       0.0364     0.1947     4.6313   253.7736            0.0983

🚨 Potential Outliers (>99th percentile):
ctr                 : 70 values > 0.0322
cvr                 : 70 values > 0.1887
cpc                 : 70 values > 2.7635
cpa                 : 70 values > 113.7673
engagement_score    : 70 values > 0.0894


In [7]:
# Categorical distributions
print("🏷️  Platform & Format Distribution:")
print("Platform distribution:")
print(df['platform_facebook'].mean(), "Facebook", df['platform_instagram'].mean(), "Instagram")

print(f"\nAd Format distribution:")
print(df['ad_format_image'].mean(), "Image", df['ad_format_video'].mean(), "Video")

print(f"\nCategory distribution:")
categories = [col for col in df.columns if col.startswith('category_') and not col.endswith('_encoded')]
category_counts = {col.replace('category_', ''): df[col].sum() for col in categories[:5]}  # Top 5 only
for cat, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{cat}: {count} ({count/len(df):.1%})")

🏷️  Platform & Format Distribution:
Platform distribution:
0.08342857142857144 Facebook 0.0015714285714285715 Instagram

Ad Format distribution:
0.9164285714285715 Image 0.08357142857142857 Video

Category distribution:
automotive: 127 (1.8%)
business: 4 (0.1%)
agriculture: 1 (0.0%)
agriculture/ecommerce: 1 (0.0%)
beauty: 1 (0.0%)


In [8]:
# Text content analysis 
print("💭 Text Content Patterns:")

content_features = ['headline_has_numbers', 'headline_has_percent', 'headline_urgency', 
                   'headline_positive', 'body_urgency', 'body_positive']

for feature in content_features:
    pct = df[feature].mean()
    high_perf_pct = df[df['high_performer']==1][feature].mean()
    low_perf_pct = df[df['high_performer']==0][feature].mean()
    
    print(f"{feature:20}: Overall={pct:.1%}, High={high_perf_pct:.1%}, Low={low_perf_pct:.1%}")
    
print(f"\n📈 Key Insights:")
print(f"• {df['headline_urgency'].mean():.1%} of headlines contain urgency words")
print(f"• {df['headline_has_numbers'].mean():.1%} of headlines contain numbers")
print(f"• {df['headline_positive'].mean():.1%} of headlines contain positive sentiment words")

💭 Text Content Patterns:
headline_has_numbers: Overall=12.1%, High=11.5%, Low=12.3%
headline_has_percent: Overall=1.8%, High=1.0%, Low=2.1%
headline_urgency    : Overall=5.1%, High=5.0%, Low=5.2%
headline_positive   : Overall=7.7%, High=8.6%, Low=7.4%
body_urgency        : Overall=14.5%, High=16.1%, Low=14.0%
body_positive       : Overall=12.5%, High=13.8%, Low=12.0%

📈 Key Insights:
• 5.1% of headlines contain urgency words
• 12.1% of headlines contain numbers
• 7.7% of headlines contain positive sentiment words


In [11]:
# Missing values and data quality check
print("🔍 Data Quality Check:")
print(f"Missing values per feature (top 10):")
missing_counts = df.isnull().sum().sort_values(ascending=False)
print(missing_counts.head(10))

print(f"\nFeatures with >0 missing values: {(missing_counts > 0).sum()}")

# Check for any constant features (could cause ML issues)
constant_features = []
for col in df.columns:
    try:
        if df[col].nunique() == 1:
            constant_features.append(col)
    except (ValueError, TypeError):
        # Skip problematic columns (like object arrays)
        continue
        
if constant_features:
    print(f"\n⚠️  Constant features (remove before ML): {constant_features}")
else:
    print(f"\n✅ No constant features found")
    
print(f"\n🎯 Dataset ready for ML training!")

🔍 Data Quality Check:
Missing values per feature (top 10):
headline_len                                                                                                                                                                                   0
category_home improvement/electrical/plumbing/painting/carpentry/furniture/sanitation/sewage/extermination/insect extermination/home services/gas appliances/power tools/building materials    0
category_health and fitness                                                                                                                                                                    0
category_healthcare                                                                                                                                                                            0
category_healthcare, education                                                                                                                                           